<a href="https://colab.research.google.com/github/HackElite-FYP/Legal-Research-Platform-Core/blob/feature%2Fretrieval/retrieval/sparse_vector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# install dependencies
!pip install torch transformers sentence-transformers
!pip install pinecone pandas tqdm

In [2]:
# init variables
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

DOC_TYPE = "case"
DOC_YEAR = "2024"

# PROCESSING_FILE_PATH = f'/content/drive/MyDrive/FYP/json/{DOC_TYPE}s_{DOC_YEAR}_v2.json'
PROCESSING_FILE_PATH = f'{DOC_TYPE}s_{DOC_YEAR}_v2.json'
PROCESSING_CASE_INDEX = 0
MODELS = ["naver/splade-cocondenser-ensembledistil"]
PINECONE_API_KEY = "pcsk_2it9oG_RzRNfQdLGg9jUen7wW6viE9JpLRgVTHtWbTZhomuZKmhuyYnrh8GgMyrHJMz37Q"
INDEX_NAME = f"law-{DOC_TYPE}s-sparse"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
!wget -O {DOC_TYPE}s_{DOC_YEAR}_v2.json https://drive.usercontent.google.com/download?id=1meeq_pyxndS9W7mmavcX_srExlpYi5Hw&export=download&authuser=3

--2025-07-22 09:09:32--  https://drive.usercontent.google.com/download?id=1meeq_pyxndS9W7mmavcX_srExlpYi5Hw
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.251.184.132, 2607:f8b0:4001:c66::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.251.184.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 30688975 (29M) [application/octet-stream]
Saving to: ‘cases_2024_v2.json’

cases_2024_v2.json  100%[===================>]  29.27M   117MB/s    in 0.3s    

2025-07-22 09:09:39 (117 MB/s) - ‘cases_2024_v2.json’ saved [30688975/30688975]



In [6]:
# load to dataframes
import json
import pandas as pd

with open(PROCESSING_FILE_PATH, 'r', encoding='utf-8') as f:
    cases_data = json.load(f)

cases_df = pd.DataFrame(cases_data)
print(cases_df.head())

                                     id  type amendmentTo  \
0  8fb1373c-2b19-4df4-bbcc-40b0720dc9b3  case               
1  f377d509-ea0d-4a2f-9e7a-4d2a14515d19  case               
2  3f222e28-d9a0-452e-aa98-498d5656613e  case               
3  9eaa002e-2133-40a3-97ec-3bf16d5829f8  case               
4  2d8f6172-6e23-4841-8b6e-70952dfc1cdc  case               

                                       filename primaryLang             title  \
0           hcc_0384_18_final_judgement_pdf.pdf   en (0.75)    CA/HCC/0384/18   
1                           wrt_0471_19_pdf.pdf   en (0.80)  CA/WRT/0471/2019   
2        ca_phc_0066_12_final_judgement_pdf.pdf   en (0.71)  CA/PHC/0066/2012   
3           cpa_0132_23_final_judgement_pdf.pdf   en (0.80)      CPA/132/2023   
4  court_of_appeal_judgment_hcc_0184_17_pdf.pdf   en (0.75)          Untitled   

                                         cleanedText  \
0  : P. Kumararatnam, J. Counsel : I. B. S. Harsh...   
1  S. U. B. KARALLIYADDE, J. Couns

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from tqdm import tqdm
import json
import nltk
from nltk.tokenize import sent_tokenize

nltk.download("punkt")
nltk.download("punkt_tab")

MODEL_NAME = "naver/splade-cocondenser-ensembledistil"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
model.eval()

MODEL_MAX_LENGTH = 512

def sentence_aware_chunking(text, max_tokens=MODEL_MAX_LENGTH): # Use MODEL_MAX_LENGTH for chunking
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_tokens = 0
    chunk_counter = 0
    for sentence in sentences:
        sentence_tokens = tokenizer.encode(sentence)
        token_count = len(sentence_tokens)
        # Adjust chunking logic to respect model's max length
        if current_tokens + token_count <= max_tokens:
            current_chunk.append(sentence)
            current_tokens += token_count
        else:
          if current_chunk:
                chunk_text = " ".join(current_chunk)
                chunks.append({
                    "chunk_index": chunk_counter,
                    "text": chunk_text,
                    "word_count": len(chunk_text.split())
                })
                chunk_counter += 1
          # Start a new chunk with the current sentence only if it is within the max_tokens limit
          current_chunk = [sentence]
          current_tokens = token_count

    if current_chunk:
        chunk_text = " ".join(current_chunk)
        chunks.append({
            "chunk_index": chunk_counter,
            "text": chunk_text,
            "word_count": len(chunk_text.split())
        })
    return chunks

def section_aware_chunking(sections, max_tokens=MODEL_MAX_LENGTH):
    chunks = []
    section_chunks_counter = 0

    for section in sections:
        section_chunks = sentence_aware_chunking(section['text'])

        for chunk in section_chunks:
            chunks.append({
                "section_index": section_chunks_counter,
                "text": chunk['text'],
                "word_count": chunk['word_count'],
                "chunk_index": chunk['chunk_index'],
                "chunks_count": len(section_chunks)
            })
            section_chunks_counter += 1

    return chunks


# Chunk all cases first (to maximize efficiency)
all_chunks = []
df_to_process = cases_df
for i in range(len(df_to_process)):
    if DOC_TYPE == "act":
        chunks = section_aware_chunking(cases_df.loc[i, 'structuredSections'])
    else:
        chunks = sentence_aware_chunking(cases_df.loc[i, 'translated'])

    chunks_count = len(chunks)
    for chunk in chunks:
        meta = {
            "filename": cases_df.loc[i, 'filename'],
            "title": cases_df.loc[i, 'title'],
            "chunks_count": chunk['chunks_count'] if 'chunks_count' in chunk else chunks_count,
            "doc_id": cases_df.loc[i, 'id']
        }

        if DOC_TYPE == "act":
            meta['section_count'] = len(cases_df.loc[i, 'structuredSections'])
            meta['section_index'] = chunk['section_index']

        all_chunks.append((chunk, meta))

print(f"\n\nProcessed Total of {len(all_chunks)} chunks in {len(df_to_process)} documents")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (532 > 512). Running this sequence through the model will result in indexing errors




Processed Total of 4888 chunks in 530 documents


In [ ]:
# worker.py
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from tqdm.notebook import tqdm

MODEL_NAME = "naver/splade-cocondenser-ensembledistil"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
model.eval()

def splade_worker(args):
    chunk, meta = args # Unpack the arguments
    inputs = tokenizer(chunk['text'], return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        outputs = model(**inputs).logits.squeeze(0)
    scores = torch.log(1 + torch.relu(outputs))
    max_scores, _ = torch.max(scores, dim=0)
    non_zero_indices = torch.nonzero(max_scores).squeeze(1).tolist()
    non_zero_values = max_scores[non_zero_indices].tolist()
    tokens = tokenizer.convert_ids_to_tokens(non_zero_indices)
    indices = tokenizer.convert_tokens_to_ids(tokens)
    # print(f"\n==>Processing chunk:{chunk['chunk_index']} | doc:{chunk['doc_id']} | {len(indices)} indices | {len(tokens)} tokens")
    data = {
        "id": f"{chunk['doc_id']}_chunk{chunk['chunk_index']}",
        "sparse_values": {"indices": indices, "values": non_zero_values},
        "metadata": {
            "chunk_index": chunk['chunk_index'],
            "filename": meta['filename'],
            "title": meta['title'],
            "text": chunk['text'][:70],
            "word_count": chunk['word_count'],
            "chunks_count": meta['chunks_count'],
            # "model": MODEL_NAME,
            # "tokens": tokens
        }
    }

    if DOC_TYPE == "act":
        data['id'] = f"{chunk['doc_id']}_section{chunk['section_index']}_chunk{chunk['chunk_index']}"
        data['metadata']['section_index'] = chunk['section_index']
        data['metadata']['section_count'] = meta['section_count']
    return data

# multiprocessing
# from multiprocessing import Pool

# with Pool(processes=8) as pool:
#     results = []
#     # Changed f to splade_worker
#     for result in tqdm(pool.imap(splade_worker, all_chunks), total=len(all_chunks)):
#         results.append(result)
#     # results = list(tqdm(pool.starmap(splade_worker, all_chunks), total=len(all_chunks)))
#     final_df = pd.DataFrame(results)

# concurrent
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.notebook import tqdm
import pandas as pd

with ProcessPoolExecutor(max_workers=8) as executor:
    # If your splade_worker takes 1 argument:
    futures = [executor.submit(splade_worker, args) for args in all_chunks[:1000]]
    results = []
    for f in tqdm(as_completed(futures), total=len(futures)):
        result = f.result()
        results.append(result)
        print(f"\n==>Processed chunk:{result['metadata']['chunk_index']} | doc:{result['id']} | {len(result['sparse_values']['indices'])} indices | {len(result['sparse_values']['values'])} values")
    final_df = pd.DataFrame(results)


# sequential
# for result in tqdm(map(splade_worker, all_chunks), total=len(all_chunks)):
#     sparse_vectors.append(result)

# print(json.dumps(sparse_vectors[0]))
print(f"\n\nProcessed Total of {len(results)} chunks in {len(df_to_process)} documents")

  0%|          | 0/1000 [00:01<?, ?it/s]

Process ForkProcess-20:


In [ ]:
#init pinecorn
from pinecone import Pinecone

# Initialize Pinecone
pc = Pinecone(
    api_key=PINECONE_API_KEY,     # Replace with your Pinecone API key
)

# Connect to index
# dense_index = pc.Index(f"{INDEX_NAME}-dense")
sparse_index = pc.Index(f"{INDEX_NAME}")

# check connection
sparse_index.describe_index_stats()

{'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'': {'vector_count': 494}},
 'total_vector_count': 494,
 'vector_type': 'sparse'}

In [ ]:
BATCH_SIZE = 1000

for i in tqdm(range(0, len(results), BATCH_SIZE)):
    # find end of batch
    i_end = min(i+BATCH_SIZE, len(results))
    # create batch
    batch = results[i:i_end]
    # upsert batch
    sparse_index.upsert(
        vectors=batch
    )

print(f"Upserted {len(results)} vectors")

Upserted 2888 vectors


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from tqdm import tqdm
import json
import nltk
from nltk.tokenize import sent_tokenize
from concurrent.futures import ProcessPoolExecutor, as_completed

nltk.download("punkt")
nltk.download("punkt_tab")

MODEL_NAME = "naver/splade-cocondenser-ensembledistil"
MODEL_MAX_LENGTH = 256 # Add model max length

# Load SPLADE model
model_name = MODELS[0]
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)
model.eval()

def sentence_aware_chunking(text, doc_id, max_tokens=MODEL_MAX_LENGTH): # Use MODEL_MAX_LENGTH for chunking
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_tokens = 0
    chunk_counter = 0
    for sentence in sentences:
        sentence_tokens = tokenizer.encode(sentence)
        token_count = len(sentence_tokens)
        # Adjust chunking logic to respect model's max length
        if current_tokens + token_count <= max_tokens:
            current_chunk.append(sentence)
            current_tokens += token_count
        else:
          if current_chunk:
                chunk_text = " ".join(current_chunk)
                chunks.append({
                    "chunk_index": chunk_counter,
                    "doc_id": doc_id,
                    "text": chunk_text,
                    "word_count": len(chunk_text.split())
                })
                chunk_counter += 1
          # Start a new chunk with the current sentence only if it is within the max_tokens limit
          current_chunk = [sentence]
          current_tokens = token_count


    if current_chunk:
        chunk_text = " ".join(current_chunk)
        chunks.append({
            "chunk_index": chunk_counter,
            "doc_id": doc_id,
            "text": chunk_text,
            "word_count": len(chunk_text.split())
        })
    return chunks


# SPLADE sparse vector generator
def splade_sparse_vector(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MODEL_MAX_LENGTH)
    with torch.no_grad():
        outputs = model(**inputs).logits.squeeze(0)  # [seq_len, vocab_size]
    scores = torch.log(1 + torch.relu(outputs))
    max_scores, _ = torch.max(scores, dim=0)  # Max-pooling across sequence
    non_zero_indices = torch.nonzero(max_scores).squeeze(1).tolist()
    non_zero_values = max_scores[non_zero_indices].tolist()
    tokens = tokenizer.convert_ids_to_tokens(non_zero_indices)
    indices = tokenizer.convert_tokens_to_ids(tokens)
    return {
        "indices": indices,
        "values": non_zero_values
    }, tokens

def process_chunk(args):
    chunk, meta = args
    sparse, tokens = splade_sparse_vector(chunk['text'])
    return {
        "id": chunk['doc_id'],
        "sparse_values": sparse,
        "metadata": {
            "chunk_index": chunk['chunk_index'],
            "filename": meta['filename'],
            "title": meta['title'],
            "text": chunk['text'][:200],
            "word_count": chunk['word_count'],
            "chunks_count": meta['chunks_count'],
            "model": MODEL_NAME,
            "tokens": tokens
        }
    }

# Chunk all cases first (to maximize efficiency)
all_chunks = []
df_to_process = cases_df
for i in range(len(df_to_process)):
    chunks = sentence_aware_chunking(cases_df.loc[i, 'translated'], cases_df.loc[i, 'id'])
    chunks_count = len(chunks)
    for chunk in chunks:
        meta = {
            "filename": cases_df.loc[i, 'filename'],
            "title": cases_df.loc[i, 'title'],
            "chunks_count": chunks_count
        }
        all_chunks.append((chunk, meta))

# Process in parallel
sparse_vectors = []
with ProcessPoolExecutor(max_workers=16) as executor:
    results = list(tqdm(executor.map(process_chunk, all_chunks), total=len(all_chunks)))
    # Filter out None values if any sentences were skipped
    sparse_vectors.extend([r for r in results if r is not None])


# Print one example
# print(json.dumps(sparse_vectors[0]))
print(f"\n\nProcessed Total of {len(sparse_vectors)} chunks in {len(df_to_process)} documents")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


KeyboardInterrupt: 

In [ ]:
from sentence_transformers import SentenceTransformer

# Create hybrid records
dense_upserts = []
sparse_upserts = []

for index, df in tqdm(cases_df.iterrows()):
    doc_id = df.id
    # dense_model = SentenceTransformer("intfloat/e5-base-v2")
    # dense_vec = dense_model.encode(df.text).tolist()
    sparse_vec = splade_sparse_vector(df.text)

     # Convert 'indices' to a list of integers
    sparse_vec['indices'] = tokenizer.convert_tokens_to_ids(sparse_vec['indices'])

    # dense_record = {
    #     "id": doc_id,
    #     "values": dense_vec,
    #     "metadata": {
    #         "text": df.text
    #     }
    # }
    # dense_upserts.append(dense_record)

    sparse_record = {
        "id": doc_id,
        "sparse_values": sparse_vec,
        "metadata": {
            "text": df.text,
            "filename": df.filename,
            "title": df.title
        }
    }
    sparse_upserts.append(sparse_record)

# Upload to Pinecone
dense_index.upsert(vectors=dense_upserts)
sparse_index.upsert(vectors=sparse_upserts)

2it [00:08,  4.35s/it]


{'upserted_count': 2}

In [ ]:
#search
query_text = "In the cozy appeal of the"
# query_dense = dense_model.encode(query_text).tolist()
query_sparse = splade_sparse_vector(query_text)

# dense_query_response = dense_index.query(
#     vector=query_dense,
#     top_k=5,
#     include_metadata=True
# )

# Convert 'indices' to a list of integers before querying
query_sparse['indices'] = tokenizer.convert_tokens_to_ids(query_sparse['indices'])

sparse_query_response = sparse_index.query(
    sparse_vector=query_sparse,
    top_k=5,
    include_metadata=True
)

# Print only the 'id' and 'score' for dense query results if metadata is not available
# for match in dense_query_response['matches']:
#     print(f"Score: {match['score']:.3f} | ID: {match['id']}")
#     # Attempt to access metadata and text fields, handling potential KeyError
#     try:
#         text = match['metadata']['text']
#         # You can customize how you want to display the relevant text here
#         # For example, print the first 100 characters:
#         print(f"Relevant Text: {text[100:200]}...")
#     except KeyError:
#         print("Metadata or text field not available for this match.")

# Print only the 'id' and 'score' for sparse query results if metadata is not available
for match in sparse_query_response['matches']:
    print(f"Score: {match['score']:.3f} | ID: {match['id']}")
    # Attempt to access metadata and text fields, handling potential KeyError
    try:
        text = match['metadata']['text']
        # You can customize how you want to display the relevant text here
        # For example, print the first 100 characters:
        print(f"Relevant Text: {text[100:200]}...")
    except KeyError:
        print("Metadata or text field not available for this match.")

Score: 0.675 | ID: 313e5fae-828a-40d0-ac21-489a6e6d05d4
Relevant Text: A  
In the matter of an 
Application in terms of 
Article 140 of the 
Constitution for mandates in 
...
Score: 0.671 | ID: 803f669e-14b7-4154-9edf-ab291a47d21e
Relevant Text:  
 
 
 
 
 
 
 
 
 
 
 
  In the matter of an Application for Writs in the 
nature of Writ of Certio...
Score: 7.160 | ID: 803f669e-14b7-4154-9edf-ab291a47d21e
Relevant Text:  
 
 
 
 
 
 
 
 
 
 
 
  In the matter of an Application for Writs in the 
nature of Writ of Certio...
Score: 5.659 | ID: 313e5fae-828a-40d0-ac21-489a6e6d05d4
Relevant Text: A  
In the matter of an 
Application in terms of 
Article 140 of the 
Constitution for mandates in 
...
